In [ ]:
import trimesh, glob, pathlib, random

In [ ]:
# Get all collision obj files
all_files = set(glob.glob(r"D:\BEHAVIOR-1K\asset_pipeline\artifacts\aggregate\objects\*\*\shape\collision\*.obj"))
numbered_files = set(glob.glob(r"D:\BEHAVIOR-1K\asset_pipeline\artifacts\aggregate\objects\*\*\shape\collision\*.*.obj"))
root_files = sorted(pathlib.Path(x) for x in all_files-numbered_files)
print(root_files)

In [ ]:
import logging
logger = logging.getLogger("trimesh")
logger.setLevel(logging.ERROR)

In [ ]:
import numpy as np
import trimesh.voxel.creation
import io

def voxelize(m):
    pitch = 0.01
    return trimesh.voxel.creation.local_voxelize(
        m, pitch=pitch, point=np.array([0, 0, 0]),
        radius=int(np.ceil(0.5/pitch))
    ).matrix

def compute_iou(visual_mesh, collision_mesh):
    v_collision = voxelize(hull_count_mesh)

    # Compute their intersection volume
    intersection = v_max_mesh & v_hull_count_mesh
    intersection_cnt = np.count_nonzero(intersection)
    union = v_max_mesh | v_hull_count_mesh
    union_cnt = np.count_nonzero(union)
    iou = intersection_cnt / union_cnt

    memory[hull_count] = (hull_count_mesh, iou)

    return iou

def load_analyze_object(client, root_fn):
    # Load the visual mesh
    visual_fn = pathlib.Path(str(root_fn).replace("collision", "visual"))
    assert visual_fn.exists(), str(visual_fn)
    with open(visual_fn, "rb") as f:
        visual_content = f.read()
    
    contents = []
    for hull_count in [2, 4, 8, 16, 32, 64]:
        hull_count_fn = root_fn.with_suffix(f".{hull_count}.obj")
        assert hull_count_fn.exists(), str(hull_count_fn)
        with open(hull_count_fn, "rb") as f:
            contents.append(f.read())
            
    # Now call analyze object
    return client.submit(analyze_object, contents, visual_content, key=str(root_fn)).result()

In [ ]:
from dask.distributed import Client

dask_client = Client('sc.stanford.edu:35423', direct_to_workers=True)

In [ ]:
from concurrent import futures
import tqdm

all_futures = {}
results = {}
randomized_root_files = list(root_files)
random.shuffle(randomized_root_files)
executor = futures.ThreadPoolExecutor(max_workers=100)
for rfn in tqdm.tqdm(randomized_root_files[:1000]):
    all_futures[executor.submit(load_analyze_object, dask_client, rfn)] = rfn

In [ ]:
print(sum(1 for f in all_futures.keys() if f.done()), "/", len(all_futures))

In [ ]:
# Temporarily process returned futures
results = {rfn: f.result() for f, rfn in all_futures.items() if f.done() and not f.exception()}
vals = np.array(list(results.values()))
x = np.array([2, 4, 8, 16, 32, 64])
y = np.mean(vals, axis=0)
std = np.std(vals, axis=0)

import matplotlib.pyplot as plt
plt.scatter(x, y)
plt.xscale("log", base=2)
plt.errorbar(x, y, std)
plt.show()

In [ ]:
thresholds = [0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6]
for threshold in thresholds:
    usable_option = np.argmax(np.concatenate([vals, np.ones(len(vals))[:, None]], axis=1) >= threshold, axis=1)
    usable_option[usable_option==6] = 5
    x_vals = x[usable_option]
    plt.hist(x_vals, bins=x)  # arguments are passed to np.histogram
    plt.xscale("log", base=2)
    plt.title(f"Thresholding at IOU >= {threshold}")
    plt.show()
    print(threshold, np.median(x_vals), np.mean(x_vals), np.std(x_vals))

In [ ]:
vals

In [ ]:
if False:
    for future in all_futures.keys():
        if future.done():
        try:
            result = future.result(timeout=0)
            rfn = all_futures[future]
            results[rfn] = result
        except:
            print("\n", all_futures[future], " ran into an error:")
            print(traceback.format_exc())